#Setup

##NDL

In [ ]:
import pandas as pd

In [ ]:
def simplifyID(id):
  return id.split('_')[0]

In [ ]:
spaNotDomLingDF = pd.read_csv('SPA_NotDom_LinguisticFeatures.csv')
spaNotDomLingDF.shape

(23, 239)

Propositional density: "Percentage of words labeled with the following part of speech tags: verbs, adjectives, adverbs, prepositions, and conjunctions / number of words"

['VERB', 'ADJ', 'ADV', 'ADP', 'CCONJ', 'SCONJ']

In [ ]:
spaNotDomLingDF.columns

Index(['Sample_ID', 'file', '# of words', '# of sentences',
       'Average sentence length', 'POS_COUNT:ADJ', 'POS_COUNT:ADP',
       'POS_COUNT:ADV', 'POS_COUNT:AUX', 'POS_COUNT:CCONJ',
       ...
       '# gender agreement errors', '# gender agreement errors: only DET',
       '# gender agreement errors per 100 nouns',
       '# gender agreement errors (only DET) per 100 nouns',
       '# number agreement errors', '# number agreement errors: only DET',
       '# number agreement errors per 100 nouns',
       '# number agreement errors (only DET) per 100 nouns',
       'Phonotactic Probability Unigram Mean',
       'Phonotactic Probability Bigram Mean'],
      dtype='object', length=239)

In [ ]:
propositonalDensityFeatsOfInterest = ['file', 'POS_Proportion:VERB','POS_Proportion:ADJ',
'POS_Proportion:ADV', 'POS_Proportion:ADP', 'POS_Proportion:SCONJ',
'POS_Proportion:CCONJ']

In [ ]:
spaNotDomPropDF = spaNotDomLingDF[propositonalDensityFeatsOfInterest]
spaNotDomPropDF.shape

(23, 7)

In [ ]:
spaNotDomPropDF.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ
0,BILP007_PostTx_Spa_WABPicnic_Coded.flo.cex,0.166667,0.011905,0.095238,0.095238,0.011905,0.035714
1,BILP008_PreTx_Spa_WABPicnic_Coded.flo.cex,0.154696,0.022099,0.077348,0.110497,0.016575,0.044199
2,BILP010_PreTx_Spa_WABPicnic_Coded.flo.cex,0.144330,0.020619,0.113402,0.082474,0.010309,0.020619
3,BILP011_PreTx_Spa_WABPicnic_Coded.flo.cex,0.126437,0.011494,0.045977,0.091954,0.022989,0.080460
4,BILP012_PreTx_Spa_WABPicnic_Coded.flo.cex,0.113402,0.010309,0.051546,0.061856,0.010309,0.061856


In [ ]:
catNotDomLingDF = pd.read_csv('CAT_NotDom_LinguisticFeatures.csv')
catNotDomPropDF = catNotDomLingDF[propositonalDensityFeatsOfInterest]
catNotDomPropDF.shape

(11, 7)

In [ ]:
notDomPropDF = pd.concat([spaNotDomPropDF, catNotDomPropDF])
notDomPropDF.shape

(34, 7)

##Statistical test for NDL

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
from matplotlib.pyplot import figure
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
def getClass(filename):
  if 'BISE' in filename:
    return 'nfv'
  if 'BILP' in filename:
    return 'lv'
  if 'BIOBS003' in filename or 'BIOBS005' in filename:
    return 'lv'
  if 'BIOBS004' in filename:
    return 'nfv'

In [ ]:
notDomPropDF.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ
0,BILP007_PostTx_Spa_WABPicnic_Coded.flo.cex,0.166667,0.011905,0.095238,0.095238,0.011905,0.035714
1,BILP008_PreTx_Spa_WABPicnic_Coded.flo.cex,0.154696,0.022099,0.077348,0.110497,0.016575,0.044199
2,BILP010_PreTx_Spa_WABPicnic_Coded.flo.cex,0.144330,0.020619,0.113402,0.082474,0.010309,0.020619
3,BILP011_PreTx_Spa_WABPicnic_Coded.flo.cex,0.126437,0.011494,0.045977,0.091954,0.022989,0.080460
4,BILP012_PreTx_Spa_WABPicnic_Coded.flo.cex,0.113402,0.010309,0.051546,0.061856,0.010309,0.061856


In [ ]:
dataY = notDomPropDF['file'].apply(getClass)
dataY.shape

(34,)

In [ ]:
from collections import Counter
Counter(dataY)

Counter({'lv': 24, 'nfv': 10})

In [ ]:
data_Y = [0 if y == 'nfv' else 1 for y in dataY ]
print(data_Y)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


In [ ]:
temp_data_X = notDomPropDF.copy()
temp_data_X['Class'] = dataY
temp_data_X.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ,Class
0,BILP007_PostTx_Spa_WABPicnic_Coded.flo.cex,0.166667,0.011905,0.095238,0.095238,0.011905,0.035714,lv
1,BILP008_PreTx_Spa_WABPicnic_Coded.flo.cex,0.154696,0.022099,0.077348,0.110497,0.016575,0.044199,lv
2,BILP010_PreTx_Spa_WABPicnic_Coded.flo.cex,0.144330,0.020619,0.113402,0.082474,0.010309,0.020619,lv
3,BILP011_PreTx_Spa_WABPicnic_Coded.flo.cex,0.126437,0.011494,0.045977,0.091954,0.022989,0.080460,lv
4,BILP012_PreTx_Spa_WABPicnic_Coded.flo.cex,0.113402,0.010309,0.051546,0.061856,0.010309,0.061856,lv


In [ ]:
lvDF = temp_data_X[temp_data_X['Class'] == 'lv']
nfvDF = temp_data_X[temp_data_X['Class'] == 'nfv']

In [ ]:
data_X = temp_data_X.drop(columns=['Class'])
data_X.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ
0,BILP007_PostTx_Spa_WABPicnic_Coded.flo.cex,0.166667,0.011905,0.095238,0.095238,0.011905,0.035714
1,BILP008_PreTx_Spa_WABPicnic_Coded.flo.cex,0.154696,0.022099,0.077348,0.110497,0.016575,0.044199
2,BILP010_PreTx_Spa_WABPicnic_Coded.flo.cex,0.144330,0.020619,0.113402,0.082474,0.010309,0.020619
3,BILP011_PreTx_Spa_WABPicnic_Coded.flo.cex,0.126437,0.011494,0.045977,0.091954,0.022989,0.080460
4,BILP012_PreTx_Spa_WABPicnic_Coded.flo.cex,0.113402,0.010309,0.051546,0.061856,0.010309,0.061856


In [ ]:
normallyDistributedFeats = []
notNormallyDistributedFeats = []
featurePValues = {}
for feature in data_X.columns:
  #use Shapiro-Wilk test to see if distribution of variables in both groups is normal
  if feature == 'file':
    continue
  _,p_nfv = stats.shapiro(nfvDF[feature])
  _,p_lv = stats.shapiro(lvDF[feature])
  # if both distribution are normal, use t-test
  if p_nfv > 0.05 and p_lv > 0.05:
    t_stat, p_feature = stats.ttest_ind(nfvDF[feature], lvDF[feature])
    featurePValues[feature] = float(p_feature)
    normallyDistributedFeats.append(feature)
  else: #if either distribution wasn't normal, use mannwhitney u test
    t_stat, p_feature = stats.mannwhitneyu(nfvDF[feature], lvDF[feature])
    featurePValues[feature] = float(p_feature)
    notNormallyDistributedFeats.append(feature)

In [ ]:
print(normallyDistributedFeats)

['POS_Proportion:VERB', 'POS_Proportion:ADJ', 'POS_Proportion:ADP', 'POS_Proportion:CCONJ']


In [ ]:
print(notNormallyDistributedFeats)

['POS_Proportion:ADV', 'POS_Proportion:SCONJ']


In [ ]:
print(featurePValues)

{'POS_Proportion:VERB': 0.48777642977056823, 'POS_Proportion:ADJ': 0.49424616613462047, 'POS_Proportion:ADV': 0.0009423724051516163, 'POS_Proportion:ADP': 0.6594773946099347, 'POS_Proportion:SCONJ': 0.26169512718710386, 'POS_Proportion:CCONJ': 0.4138150687119513}


In [ ]:
from statsmodels.stats.multitest import fdrcorrection

# 2. Apply FDR correction
all_features = list(featurePValues.keys())
raw_pvals = [featurePValues[feat] for feat in all_features]
rejected, fdr_corrected_pvals = fdrcorrection(raw_pvals)

# 3. Create corrected p-value dictionary
featureFDRPValues = dict(zip(all_features, fdr_corrected_pvals))


In [ ]:
print(featureFDRPValues)

{'POS_Proportion:VERB': np.float64(0.5930953993615445), 'POS_Proportion:ADJ': np.float64(0.5930953993615445), 'POS_Proportion:ADV': np.float64(0.005654234430909698), 'POS_Proportion:ADP': np.float64(0.6594773946099347), 'POS_Proportion:SCONJ': np.float64(0.5930953993615445), 'POS_Proportion:CCONJ': np.float64(0.5930953993615445)}


In [ ]:
#features with significant (i.e p<0.05)
significantFeatures = {k: v.item() for k, v in featureFDRPValues.items() if v < 0.05}
significantFeatures

{'POS_Proportion:ADV': 0.005654234430909698}

##DL

In [ ]:
spaDomLingDF = pd.read_csv('SPA_Dom_LinguisticFeatures.csv')
spaDomPropDF = spaDomLingDF[propositonalDensityFeatsOfInterest]
spaDomPropDF.shape

(11, 7)

In [ ]:
catDomLingDF = pd.read_csv('CAT_Dom_LinguisticFeatures.csv')
catDomPropDF = catDomLingDF[propositonalDensityFeatsOfInterest]
catDomPropDF.shape

(23, 7)

In [ ]:
domPropDF = pd.concat([spaDomPropDF, catDomPropDF])
domPropDF.shape

(34, 7)

##Statistical test for DL

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
from matplotlib.pyplot import figure
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
def getClass(filename):
  if 'BISE' in filename:
    return 'nfv'
  if 'BILP' in filename:
    return 'lv'
  if 'BIOBS003' in filename or 'BIOBS005' in filename:
    return 'lv'
  if 'BIOBS004' in filename:
    return 'nfv'

In [ ]:
domPropDF.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ
0,BILP006_PreTx_Spa_WABPicnic_Coded.flo.cex,0.124378,0.019900,0.159204,0.097015,0.029851,0.039801
1,BILP009_PreTx_Spa_WABPicnic_Coded.flo.cex,0.140625,0.015625,0.093750,0.070312,0.023438,0.101562
2,BILP013_PreTx_Spa_WABPicnic_Coded.flo.cex,0.109635,0.039867,0.073090,0.106312,0.016611,0.079734
3,BILP022_PreTx_Spa_WABPicnic_Coded.flo.cex,0.120567,0.007092,0.092199,0.127660,0.042553,0.063830
4,BILP024_BACC016_PicnicScene_Spa_Pre_20240130_C...,0.145349,0.017442,0.098837,0.075581,0.023256,0.075581


In [ ]:
dataY = domPropDF['file'].apply(getClass)
dataY.shape

(34,)

In [ ]:
from collections import Counter
Counter(dataY)

Counter({'lv': 24, 'nfv': 10})

In [ ]:
data_Y = [0 if y == 'nfv' else 1 for y in dataY ]
print(data_Y)

[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0]


In [ ]:
temp_data_X = domPropDF.copy()
temp_data_X['Class'] = dataY
temp_data_X.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ,Class
0,BILP006_PreTx_Spa_WABPicnic_Coded.flo.cex,0.124378,0.019900,0.159204,0.097015,0.029851,0.039801,lv
1,BILP009_PreTx_Spa_WABPicnic_Coded.flo.cex,0.140625,0.015625,0.093750,0.070312,0.023438,0.101562,lv
2,BILP013_PreTx_Spa_WABPicnic_Coded.flo.cex,0.109635,0.039867,0.073090,0.106312,0.016611,0.079734,lv
3,BILP022_PreTx_Spa_WABPicnic_Coded.flo.cex,0.120567,0.007092,0.092199,0.127660,0.042553,0.063830,lv
4,BILP024_BACC016_PicnicScene_Spa_Pre_20240130_C...,0.145349,0.017442,0.098837,0.075581,0.023256,0.075581,lv


In [ ]:
lvDF = temp_data_X[temp_data_X['Class'] == 'lv']
nfvDF = temp_data_X[temp_data_X['Class'] == 'nfv']

In [ ]:
data_X = temp_data_X.drop(columns=['Class'])
data_X.head()

,file,POS_Proportion:VERB,POS_Proportion:ADJ,POS_Proportion:ADV,POS_Proportion:ADP,POS_Proportion:SCONJ,POS_Proportion:CCONJ
0,BILP006_PreTx_Spa_WABPicnic_Coded.flo.cex,0.124378,0.019900,0.159204,0.097015,0.029851,0.039801
1,BILP009_PreTx_Spa_WABPicnic_Coded.flo.cex,0.140625,0.015625,0.093750,0.070312,0.023438,0.101562
2,BILP013_PreTx_Spa_WABPicnic_Coded.flo.cex,0.109635,0.039867,0.073090,0.106312,0.016611,0.079734
3,BILP022_PreTx_Spa_WABPicnic_Coded.flo.cex,0.120567,0.007092,0.092199,0.127660,0.042553,0.063830
4,BILP024_BACC016_PicnicScene_Spa_Pre_20240130_C...,0.145349,0.017442,0.098837,0.075581,0.023256,0.075581


In [ ]:
normallyDistributedFeats = []
notNormallyDistributedFeats = []
featurePValues = {}
for feature in data_X.columns:
  #use Shapiro-Wilk test to see if distribution of variables in both groups is normal
  if feature == 'file':
    continue
  _,p_nfv = stats.shapiro(nfvDF[feature])
  _,p_lv = stats.shapiro(lvDF[feature])
  # if both distribution are normal, use t-test
  if p_nfv > 0.05 and p_lv > 0.05:
    t_stat, p_feature = stats.ttest_ind(nfvDF[feature], lvDF[feature])
    featurePValues[feature] = float(p_feature)
    normallyDistributedFeats.append(feature)
  else: #if either distribution wasn't normal, use mannwhitney u test
    t_stat, p_feature = stats.mannwhitneyu(nfvDF[feature], lvDF[feature])
    featurePValues[feature] = float(p_feature)
    notNormallyDistributedFeats.append(feature)

In [ ]:
print(normallyDistributedFeats)

['POS_Proportion:VERB', 'POS_Proportion:ADP', 'POS_Proportion:CCONJ']


In [ ]:
print(notNormallyDistributedFeats)

['POS_Proportion:ADJ', 'POS_Proportion:ADV', 'POS_Proportion:SCONJ']


In [ ]:
print(featurePValues)

{'POS_Proportion:VERB': 0.5058149403538479, 'POS_Proportion:ADJ': 0.46977719712770516, 'POS_Proportion:ADV': 0.0003054138505670205, 'POS_Proportion:ADP': 0.31726279402758295, 'POS_Proportion:SCONJ': 0.006641776989972193, 'POS_Proportion:CCONJ': 0.12022479811832362}


In [ ]:
from statsmodels.stats.multitest import fdrcorrection

# 2. Apply FDR correction
all_features = list(featurePValues.keys())
raw_pvals = [featurePValues[feat] for feat in all_features]
rejected, fdr_corrected_pvals = fdrcorrection(raw_pvals)

# 3. Create corrected p-value dictionary
featureFDRPValues = dict(zip(all_features, fdr_corrected_pvals))

In [ ]:
print(featureFDRPValues)

{'POS_Proportion:VERB': np.float64(0.5058149403538479), 'POS_Proportion:ADJ': np.float64(0.5058149403538479), 'POS_Proportion:ADV': np.float64(0.001832483103402123), 'POS_Proportion:ADP': np.float64(0.4758941910413744), 'POS_Proportion:SCONJ': np.float64(0.01992533096991658), 'POS_Proportion:CCONJ': np.float64(0.24044959623664725)}


In [ ]:
#features with significant (i.e p<0.05)
significantFeatures = {k: v.item() for k, v in featureFDRPValues.items() if v < 0.05}
significantFeatures

{'POS_Proportion:ADV': 0.001832483103402123,
 'POS_Proportion:SCONJ': 0.01992533096991658}

#Compare ling NDL vs ling DL

In [ ]:
domPropDF['Class'] = domPropDF['file'].apply(getClass)
notDomPropDF['Class'] = notDomPropDF['file'].apply(getClass)

In [ ]:
domPropDF.shape

(34, 7)

In [ ]:
notDomPropDF.shape

(34, 7)

In [ ]:
nfvNotDomPropDF = notDomPropDF[notDomPropDF['Class'] == 'nfv']
lvNotDomPropDF = notDomPropDF[notDomPropDF['Class'] == 'lv']

In [ ]:
lvDomPropDF = domPropDF[domPropDF['Class'] == 'lv']
nfvDomPropDF = domPropDF[domPropDF['Class'] == 'nfv']
#

In [ ]:
import scipy.stats as stats
from scipy.stats import pearsonr


In [ ]:
import pandas as pd

# Example: assume your DataFrame is called df
means = temp_data_X.mean(numeric_only=True)
stds = temp_data_X.std(numeric_only=True)
mins = temp_data_X.min(numeric_only=True)
maxs = temp_data_X.max(numeric_only=True)

# Combine into a single summary table
summary = pd.DataFrame({
    'Mean': means,
    'Standard Deviation': stds,
    'Min': mins,
    'Max': maxs
})

print(summary)

In [ ]:
def statsSummary(df):
  means = df.mean(numeric_only=True)
  stds = df.std(numeric_only=True)
  mins = df.min(numeric_only=True)
  maxs = df.max(numeric_only=True)
  summary = pd.DataFrame({
    'Mean': means,
    'Standard Deviation': stds,
    'Min': mins,
    'Max': maxs
  })
  return summary

In [ ]:
def domNonDomComparison(domDF, nonDomDF):
  normallyDistributedFeats = []
  notNormallyDistributedFeats = []
  featurePValues = {}
  for feature in domDF.columns:
    #output mean, sd, min, max,

  #use Shapiro-Wilk test to see if distribution of variables in both groups is normal
    if feature in ['file', 'Class']:
      continue
    _,p_dom = stats.shapiro(domDF[feature])
    _,p_nonDom = stats.shapiro(nonDomDF[feature])
    # if both distribution are normal, use t-test
    if p_dom > 0.05 and p_nonDom > 0.05:
      t_stat, p_feature = stats.ttest_ind(domDF[feature], nonDomDF[feature])
      featurePValues[feature] = float(p_feature)
      normallyDistributedFeats.append(feature)
    else: #if either distribution wasn't normal, use mannwhitney u test
      t_stat, p_feature = stats.mannwhitneyu(domDF[feature], nonDomDF[feature])
      featurePValues[feature] = float(p_feature)
      notNormallyDistributedFeats.append(feature)
    from statsmodels.stats.multitest import fdrcorrection

# 2. Apply FDR correction
  all_features = list(featurePValues.keys())
  raw_pvals = [featurePValues[feat] for feat in all_features]
  rejected, fdr_corrected_pvals = fdrcorrection(raw_pvals)

  # 3. Create corrected p-value dictionary
  featureFDRPValues = dict(zip(all_features, fdr_corrected_pvals))
  featureFDRPValues = {k: v.item() for k, v in featureFDRPValues.items()}
  print(featureFDRPValues)
  significantFeatures = {k: v for k, v in featureFDRPValues.items() if v < 0.05}
  print("Significant features:", significantFeatures)

In [ ]:
lvNotDomPropDF.describe().T.drop(columns='count')

,mean,std,min,25%,50%,75%,max
POS_Proportion:VERB,0.146222,0.028587,0.101695,0.123827,0.149570,0.165453,0.202020
POS_Proportion:ADJ,0.019679,0.011737,0.000000,0.011802,0.019417,0.026077,0.052117
POS_Proportion:ADV,0.089897,0.035890,0.037383,0.062620,0.087920,0.110179,0.189394
POS_Proportion:ADP,0.089049,0.025157,0.048544,0.071853,0.090511,0.110651,0.141414
POS_Proportion:SCONJ,0.025026,0.022997,0.000000,0.010309,0.020820,0.029420,0.101215
POS_Proportion:CCONJ,0.063020,0.023948,0.016667,0.044946,0.063638,0.074923,0.110236


In [ ]:
lvDomPropDF.describe().T.drop(columns='count')

,mean,std,min,25%,50%,75%,max
POS_Proportion:VERB,0.166616,0.032050,0.109635,0.142299,0.163340,0.191913,0.226744
POS_Proportion:ADJ,0.019126,0.013822,0.000000,0.008591,0.018563,0.027829,0.043478
POS_Proportion:ADV,0.101203,0.034360,0.038961,0.075338,0.095419,0.125568,0.180233
POS_Proportion:ADP,0.087628,0.024323,0.043478,0.074264,0.086774,0.100773,0.146341
POS_Proportion:SCONJ,0.025990,0.014052,0.000000,0.017248,0.025355,0.039323,0.049808
POS_Proportion:CCONJ,0.062187,0.018375,0.036184,0.044960,0.062427,0.075347,0.101562


In [ ]:
domNonDomComparison(lvNotDomPropDF, lvDomPropDF)

{'POS_Proportion:VERB': 0.146756924620608, 'POS_Proportion:ADJ': 0.8929836067392565, 'POS_Proportion:ADV': 0.8122923547783844, 'POS_Proportion:ADP': 0.8929836067392565, 'POS_Proportion:SCONJ': 0.8778194611726635, 'POS_Proportion:CCONJ': 0.8929836067392565}
Significant features: {}


In [ ]:
nfvNotDomPropDF.describe().T.drop(columns='count')

,mean,std,min,25%,50%,75%,max
POS_Proportion:VERB,0.138105,0.035600,0.093617,0.118578,0.126603,0.160886,0.209302
POS_Proportion:ADJ,0.016347,0.015191,0.000000,0.000000,0.020556,0.025551,0.038298
POS_Proportion:ADV,0.040229,0.031921,0.013158,0.019207,0.027106,0.050421,0.119048
POS_Proportion:ADP,0.083545,0.047179,0.017857,0.052421,0.080936,0.123146,0.148936
POS_Proportion:SCONJ,0.015686,0.016925,0.000000,0.000000,0.012785,0.024963,0.047619
POS_Proportion:CCONJ,0.072330,0.041326,0.000000,0.039692,0.089515,0.102203,0.118644


In [ ]:
nfvDomPropDF.describe().T.drop(columns='count')

,mean,std,min,25%,50%,75%,max
POS_Proportion:VERB,0.175160,0.037696,0.120000,0.154670,0.170291,0.199286,0.247423
POS_Proportion:ADJ,0.017255,0.020572,0.000000,0.000000,0.014797,0.025867,0.065714
POS_Proportion:ADV,0.033901,0.046728,0.000000,0.003125,0.020000,0.045311,0.154639
POS_Proportion:ADP,0.076706,0.037261,0.020619,0.063542,0.072948,0.081644,0.137143
POS_Proportion:SCONJ,0.009993,0.013958,0.000000,0.000000,0.004274,0.011952,0.040000
POS_Proportion:CCONJ,0.077379,0.037552,0.030928,0.046154,0.067143,0.108662,0.137500


In [ ]:
domNonDomComparison(nfvNotDomPropDF, nfvDomPropDF)

{'POS_Proportion:VERB': 0.21875038394398466, 'POS_Proportion:ADJ': 0.8451370142718898, 'POS_Proportion:ADV': 0.8451370142718898, 'POS_Proportion:ADP': 0.8451370142718898, 'POS_Proportion:SCONJ': 0.8451370142718898, 'POS_Proportion:CCONJ': 0.8451370142718898}
Significant features: {}
